# 原理和细节

本篇集中说明 Core 的实现约定，供开发自定义节点、排查运行行为和扩展缓存时查阅。日常使用先看[计算图框架](计算图框架.ipynb)、[计算引擎](计算引擎.ipynb)和[缓存](缓存.ipynb)。本篇示例也可独立执行。

- [节点生命周期与依赖值](#node-lifecycle)
- [依赖绑定与构造路径](#dependency-binding)
- [计算身份、实例身份与运行身份](#node-identity)
- [上下文状态与同步](#context-state)
- [引擎调度与并发](#engine-scheduling)
- [缓存分层与存储](#cache-storage)
- [结果键与跨运行复用边界](#result-keys)
- [计算图可视化的身份约定](#graph-visualization)

<a id="node-lifecycle"></a>

## 节点生命周期与依赖值

节点之间的连接是有向无环图（DAG）。若 A 依赖 B，请求沿 A → B 传递，结果沿 B → A 汇总。引擎在每次运行前验证目标为完整节点，并按实例身份检查循环依赖；验证本身不读取数据，也不计算 QSID。

| 方法 | 作用与返回值 |
|---|---|
| `init_compute(path, init_data, context)` | 登记状态、汇总初始化范围；返回与依赖对应的初始化数据，空列表表示停止展开该分支 |
| `prepare_compute(prepare_data, context)` | 执行已登记任务的 IO 准备；不应直接修改全局运行状态，可能在线程池中执行 |
| `forward_compute(path, fwd_data, context)` | 返回 `(依赖请求, 局部上下文)`；空请求表示不继续计算依赖 |
| `backward_compute(path, inputs, context, local_context=None)` | 使用依赖结果和当前节点的局部上下文计算返回值 |
| `compute(path, fwd_data, context)` | 基类递归编排 forward → 依赖 compute → backward |
| `merge_result(result_list, context)` | 合并数据分片结果；默认原样返回结果列表 |

`path` 是从目标到当前节点的 `InstanceID` 列表。准备阶段只处理 `context.PrepareNodeDict` 中登记的任务，单纯重写 `prepare_compute()` 不会使它自动执行。

Node 在子类定义时包装生命周期方法，将请求列表及依赖结果转换为 `DependencyValues`。该视图保留依赖槽位的结构和集合顺序，既能按名称访问，也兼容按展开位置访问。`inputs` 与旧参数名 `bwd_data_list` 都可用于后向方法。

下面演示登记一次准备任务。实际节点可以在准备方法里加载数据并写入专用缓存；示例通过普通数据空间写入一张小表。

In [1]:
import logging
from QuantStudio.Core import setDefaultLogLevel

setDefaultLogLevel(logging.WARNING)

In [2]:
import datetime as dt
import tempfile
import pandas as pd
from pydantic import Field
from QuantStudio.Core.api import Node, DependencyModel, Context, Engine
from QuantStudio.Core.FileCache import FeatherDTCache


class PreparedNumber(Node):
    """通过准备任务写入数据的叶子节点。"""

    class __QS_DepClass__(DependencyModel):
        """没有依赖。"""

    class __QS_ArgClass__(Node.__QS_ArgClass__):
        Value: float = Field(frozen=True, title="数值")

    def init_compute(self, path, init_data, context):
        """登记准备任务，返回空依赖请求。"""
        context.PrepareNodeDict[self.InstanceID] = (self.InstanceID, self.Args.Value)
        return []

    def prepare_compute(self, prepare_data, context):
        """将准备数据写入缓存。"""
        context.DataCache.writeData(self.InstanceID, pd.DataFrame({"值": [prepare_data]}),
                                    if_exists="replace")

    def backward_compute(self, path, inputs, context, local_context=None):
        """读取本轮准备的数据。"""
        return context.DataCache.readData(self.InstanceID).at[0, "值"]


prepared = PreparedNumber(args={"Value": 8})
with tempfile.TemporaryDirectory() as cache_dir:
    with FeatherDTCache(args={"CacheDir": cache_dir, "DTRuler": [dt.datetime(2025, 1, 1)]}) as cache:
        assert Engine().run([prepared], Context(DataCache=cache)) == [8]
print("准备任务已执行")

准备任务已执行


<a id="dependency-binding"></a>

## 依赖绑定与构造路径

`__QS_DepClass__` 是 Pydantic `DependencyModel` 子类，字段同时定义名称、类型、默认值、说明和集合长度约束。默认 `NodeDependencies` 只有 `nodes: tuple[Node, ...]`；命名模型使用字段映射或完整依赖模型绑定，无依赖节点可以声明空模型。

| 接口 | 含义 |
|---|---|
| `Dependencies` | 不可变的完整依赖模型，连接关系的唯一来源 |
| `Deps` | 每次按依赖边展开生成的列表副本 |
| `NodeClass.dependency_info()` | 不构造节点，直接读取槽位声明 |
| `node.dependency_info()` | 读取声明和实际绑定值 |
| `iter_dependency_edges()` | 按字段与集合顺序产生边，包含槽位路径和节点 |
| `dependency_values(values)` | 将展开值组织成与槽位对应的视图 |

同名槽位通过 `inputs.name` 或 `inputs["name"]` 读取。若槽位名与 `flat`、`items` 等视图成员同名，应使用字符串索引或 `field()`，避免属性冲突。

模板中的“未绑定”、显式 `None`、空元组不同：是否可为 `None`、是否允许空集合，取决于字段声明。部分绑定可以暂时不满足集合数量下界，但会检查成员类型和数量上界；最终 `build()` 执行完整模型的字段、模型验证器。`bind()`、`unbind()`、`with_args()` 返回新模板，并保留依赖节点引用。

### 统一构造入口与初始化钩子

`from_dependencies()` 绕过子类便利构造器，执行基类初始化后调用 `_QS_post_bind()`。`NodeTemplate.build()` 与基类 `Node.new()` 使用此入口。

需要派生初始化时，可将其放到 `_QS_post_bind()`；若子类还有便利构造器，应在 `super().__init__()` 后自行调用该钩子，因为普通 `Node.__init__()` 不会自动调用。这样模板与直接构造才能保持一致。影响计算的配置仍需纳入参数或依赖模型，不能仅存在便利构造器写入的私有属性中。

<a id="node-identity"></a>

## 计算身份、实例身份与运行身份

节点 ID：

- `QSID` 表示完整节点的计算语义，包含类、有效参数、行为版本、命名依赖和集合顺序；相同定义可在不同运行中保持相同身份。
- `InstanceID` 标识独立节点实例。构造时生成 UUID，配置相同的独立实例也具有不同 ID。`NodeDict`、`NodeState`、`NodeErrors` 以它为节点键，防止实例状态互相覆盖。

运行 ID：每次 `run()` 通过 `begin_run()` 重新生成 `Context.RunID`，同时清理上轮临时状态，保留显式资源和数据缓存。同一轮的节点共享该 ID。

`QSID` 由 `model_dump()` 的内容经确定性序列化后计算 SHA-256，并按对象机制缓存。Node 的描述在基类的类名、参数基础上加入 `__QS_BehaviorVersion__` 和依赖 QSID。基础 Node 的 `Name`、`Parallel`、`TaskExecutor` 标记为 `exclude=True`，不参与该描述；子类可以重新定义字段或扩展 `model_dump()`。

实现逻辑发生会影响结果的变更时，需要维护相应行为版本或描述，普通节点的 QSID 不会自动包含方法源码。相同 QSID 表示计算定义相同，结果还可能受请求、数据和状态影响。

In [3]:
a = PreparedNumber(args={"Value": 1, "Name": "实例甲"})
b = a.new(args={"Name": "实例乙"})
c = a.new(args={"Value": 2})
assert a.QSID == b.QSID
assert a.InstanceID != b.InstanceID
assert a.QSID != c.QSID

context = Context()
first_run = context.RunID
context.NodeState[a.InstanceID] = {"临时值": 1}
context.begin_run()
assert context.RunID != first_run
assert context.NodeState == {}
print("语义相同、实例不同；新运行隔离临时状态")

语义相同、实例不同；新运行隔离临时状态


### 执行结果身份与副作用声明

`node.execution_key(request, data_version=None)` 计算包含 QSID、请求、数据版本的 SHA-256。它只是生成标识，不执行缓存读写，也不会自动读取 Context；调用方需要传入所需的版本和请求字段。未指定数据版本时，不承诺跨运行缓存有效。

`ExecutionEffect` 提供 `PURE`（纯计算）、`READ`（外部读取）、`STATEFUL`（状态递推）、`WRITE`（写入）声明，Node 默认是 `STATEFUL`。当前核心调度不会仅凭该声明或相同 QSID 自动合并执行；两个独立写入实例不会因此合并。实际复用范围由节点及其结果键决定。

In [4]:
request = {"日期": "2025-01-01", "截面": ("A", "B")}
key = a.execution_key(request, data_version="行情-v1")
assert key == b.execution_key(request, data_version="行情-v1")
assert key != a.execution_key(request, data_version="行情-v2")
print("改变数据版本会改变执行结果键")

改变数据版本会改变执行结果键


<a id="context-state"></a>

## 上下文状态与同步

`Context` 是全局运行上下文，`LocalContext` 用于当前节点一次前向、后向调用之间传递局部信息。`DTInitData` 包含 `DTRange`；`DTLocalContext` 额外包含 `DTs`，省略 `DTRange` 时用非空 `DTs` 的首尾生成区间。

| 字段 | 结构或职责 |
|---|---|
| `Mode` | `"PRD"` 或 `"DEBUG"`，节点可据此选择诊断行为 |
| `RunID` / `DataVersion` | 本轮身份 / 调用方声明的数据快照版本 |
| `Resources` | 调用方注入的服务或句柄映射 |
| `DataCache` | 显式提供的缓存，生命周期由调用方管理 |
| `EventHandler` | 执行事件回调 |
| `NodeDict` | `{InstanceID: Node}`，初始化阶段登记 |
| `NodeState` | `{InstanceID: 节点状态}`，由具体节点维护 |
| `NodeErrors` | `{InstanceID: Exception}`，保留跳过模式的失败原因 |
| `PrepareNodeDict` | `{准备键: (InstanceID, 准备数据)}` |
| `ExtraData` | 扩展数据；部分保留键由引擎管理 |
| `PID` / `PIDList` | 当前进程任务 ID / 全部进程任务 ID，默认 `"0"` / `["0"]` |
| `SplitType` | `"连续切分"` 或 `"间隔切分"` |
| `Event` | `{同步键: Event}`；键由具体节点约定，不一定直接等于 InstanceID |
| `Sub2MainQueue` | 子进程向主进程传递消息的队列 |
| `TaskExecutor` / `MaxWorkers` | 供节点内部使用的执行器 / 并发数 |

`begin_run()` 清空 `NodeDict`、`NodeState`、`NodeErrors`、`PrepareNodeDict`、`Event`，并移除引擎管理的部分扩展项；其余用户扩展数据、显式资源与数据缓存保留。跨轮业务状态不应依赖上述临时字典。

### 分片与状态同步扩展点

基类 `Context.split(n)` 返回 `[self] * n`，只是重复引用；`LocalContext.split()` 也如此。基类 `getUpdateData()` 返回空字典，`updateContext()` 不执行更新。它们不自动分片或同步任意字段，真正行为由 `FactorContext` 等子类实现。

进程模式中的资源、回调和共享数据需要能被对应执行器传递。进程内修改不会自动反映到主进程，应由具体上下文选择同步内容；缓存也有 `getUpdateData()`、`updateCache()` 接口。

### 活动上下文与引擎栈

`with Context() as ctx:` 将上下文压入 `__QS_Context__`，退出时弹出；进入 `with` 不会开始运行。`with Engine() as engine:` 同样管理 `__QS_Engine__`。引擎栈初始包含一个默认 Engine，部分业务便捷接口读取栈顶对象；嵌套退出后恢复之前的选择。这些模块级栈不应视为线程隔离的运行状态。

In [5]:
from QuantStudio.Core.CalcEngine import StackEngine, __QS_Engine__
from QuantStudio.Core.Node import __QS_Context__, DTLocalContext

engine_depth, context_depth = len(__QS_Engine__), len(__QS_Context__)
with StackEngine() as active_engine, Context() as active_context:
    assert __QS_Engine__[-1] is active_engine
    assert __QS_Context__[-1] is active_context
assert len(__QS_Engine__) == engine_depth
assert len(__QS_Context__) == context_depth

local = DTLocalContext(DTs=[dt.datetime(2025, 1, 1), dt.datetime(2025, 1, 2)])
assert local.DTRange == (local.DTs[0], local.DTs[-1])
print("退出 with 后恢复活动对象")

退出 with 后恢复活动对象


<a id="engine-scheduling"></a>

## 引擎调度与并发

所有引擎继承 `Engine`。初始化与准备结束后进入计算阶段；准备阶段只针对已登记的任务，`IOConcurrentNum=None` 时按任务数确定线程并发数，不是固定的低并发上限。

| 阶段 | Engine | StackEngine | ParallelEngine | TreeEngine |
|---|---|---|---|---|
| 初始化 | 队列，广度优先 | 显式栈，深度优先 | 继承 Engine | 继承 Engine |
| 准备 | 顺序或线程并发 IO | 继承 Engine | 继承 Engine | 继承 Engine |
| 计算 | 按目标递归调用 Node.compute | 深度优先收集前向调用，再逆序处理后向调用 | 子进程执行目标图，主进程合并目标结果 | 传播请求后，按依赖完成情况并发提交后向任务 |

### StackEngine 的调用顺序

初始化深度优先展开；前向计算也先沿第一个依赖深入，收集节点、路径和局部上下文，之后按收集顺序逆序执行后向方法，并恢复槽位对应的依赖结果。

例如 `Root → [Left, Right]`、`Left → [Leaf]`：

```text
初始化：Root → Left → Leaf → Right
前向：  Root → Left → Leaf → Right
后向：  Right → Leaf → Left → Root
```

不能将其简单理解为 Engine 的递归后序调用；自定义节点不应依赖无明确约定的兄弟节点执行顺序来传递业务状态。

### TreeEngine 的请求任务与实例互斥

TreeEngine 先为每次依赖引用传播请求，使用目标序号和依赖展开位置组成任务 token。即使同一个节点被多个槽位引用，也分别保留各次调用的局部上下文和结果，不会只凭 QSID 去重。

调度器在子任务均完成后提交当前节点的 `backward_compute()`，通过 futures 等待完成事件；`CalcConcurrentNum` 限制同时提交的计算任务数。同一 `InstanceID` 的后向任务不会同时执行，但不同实例即使 QSID 相同也可各自执行。

线程模式共享上下文；进程模式通过初始化器保存进程上下文，再用 `getUpdateData(node_id_list=...)` / `updateContext()` 在任务边界交换选定状态。扩展的上下文需兼容这些接口。

### ParallelEngine 的分片与进程消息

任务数取 `len(context.PIDList)`。上下文调用 `split(n)`，请求若支持 `split(n, context)` 就分片，否则重复发送。声明 `__QS_ParallelBroadcast__` 的节点广播根请求，由节点内部安排分工。

每个进程执行所有目标，主进程收集后调用目标节点的 `merge_result()`。节点及上下文必须配套实现分片、结果组织和同步，不是设置多个进程 ID 就会自动切分任意数据。

| 消息 | 用途 |
|---|---|
| `Event` | 累加同步信号，所有任务就绪后设置事件 |
| `Calc` | 返回单个目标的结果 |
| `CalcError` | 返回跳过模式下目标计算异常 |
| `Done` | 返回子进程结束信息、选定上下文更新、错误及失败状态 |
| `NodeEvent` | 将执行事件交给主进程回调 |
| `TaskError` | 返回应抛出的任务异常 |

主进程检查未正常发送结束消息的异常退出，失败时终止并回收其余进程并丢弃未消费消息。共享截面等待还会检查相关依赖的跨进程失败信号，避免依赖已失败而其他进程无限等待。

### 失败传播

`FailMode="skip"` 时，节点自身异常记录为 `FAILED`，消费该失败结果的目标标为 `SKIPPED`，结果为 `None`。错误按依赖结果传播，不意味着所有兄弟分支都会跳过；不同引擎可能已执行或尚未执行这些分支。

初始化失败不继续展开该节点的依赖；准备失败使节点在计算阶段被跳过；计算失败会影响依赖它的结果。异常保存在 `NodeErrors[InstanceID]`，状态保存在 `NodeState[InstanceID]["__status__"]`。这些是失败标记，不是所有节点的完整成功状态机。

<a id="cache-storage"></a>

## 缓存分层与存储

### 接口和实现关系

```text
Cache                         通用键值读写、生命周期
└── DTCache                   增加覆盖区间记录
    ├── FileDTCache           文件组织与锁，数据格式接口仍需实现
    │   └── FeatherDTCache    Feather 格式读写
    └── FactorCache           原始数据与因子结果、进程分片接口
```

`FileFactorCache(FileDTCache, FactorCache)` 组合文件层与因子层；`FeatherFactorCache(FileFactorCache, FeatherDTCache)` 再提供 Feather 格式。因此 FileDTCache、FileFactorCache 都不是可以直接完成所有 DataFrame 读写的格式实现。

Feather 实现默认后缀 `.feather`；`data_type="object"` 时使用 `.pkl`，读写两端需保持一致。Feather 读取遇到 `ArrowInvalid` 时返回 `None`，其他读写异常仍可能向外抛出。

### 目录、锁与持久化

```text
CacheDir/
├── state.pkl                  覆盖区间等状态，默认 StateFile
├── _Cache.lock                清理操作的锁
├── Data/                      普通数据
│   ├── <key>.lock
│   ├── <key><suffix>
│   └── <key>_meta.json         可选元信息
├── DTData/                    时序数据，同样按 key 存储
├── LockFile                   因子缓存使用的锁文件
├── RawData/<PID>/             因子原始数据，按字段等继续组织
└── FactorData/<PID>/          因子结果，按结果键和字段继续组织
```

最后三项属于文件因子缓存。单项读写通过文件锁互斥，不能据此认为用户自行修改的全部 Python 对象也具备并发安全性。

启动时定位已存在的 CacheDir，未指定时使用临时目录；`"new"` 清理对应数据空间，`"continue"` 调用 `load()`。状态文件不存在时退回清理新建。正常 `end(clear=False)` 调用 `dump()`，默认将覆盖区间状态保存在 `state.pkl`。

`writeData()` / `writeDTData()` 的 append 逻辑按索引合并、去重并排序，重复索引保留旧记录；replace 则整体替换数据。不能将 append 当作修正已有值的更新操作。

### 覆盖范围记录与同步

`DTCache._CachedDTRange` 保存 `{key: DataFrame(columns=["StartDT", "EndDT"])}`，一个键可对应多个连续区间。`updateDTRange()` 登记并合并范围，`_mergeDTRange()` 根据标尺相邻性合并邻接区间。

`getDTRange()` 返回需要补算的一个区间或 `None`，不是逐项验证缓存文件完整性，也不承诺返回精确的所有离散缺口列表。底层 `writeDTData()` 不更新覆盖记录，需要调用方或业务节点显式维护。

`getUpdateData(key_list=...)` 可选择性导出覆盖记录，`updateCache()` 通过 `updateDTRange()` 合并接收到的范围。具体上下文决定何时、对哪些键同步；底层接口本身不自动同步所有运行字段。

因子缓存另外提供 `writeRawData()` / `readRawData()` 与 `writeFactorData()` / `readFactorData()`。原始数据可按 ID 列分发到进程目录；因子读取可等待其他进程数据就绪。其分片配置和业务同步机制应保持配套。

<a id="result-keys"></a>

## 结果键与跨运行复用边界

缓存接口仅接收调用方传入的字符串键，不会自行选择 QSID、InstanceID 或 RunID。基础 `Node.compute()` 没有自动结果缓存。

| 对象或场景 | 键的构成 |
|---|---|
| 手动 `cache.writeData(key, ...)` | 调用方传入的 key |
| 节点运行状态 | 外层使用 InstanceID，内部由节点按请求组织 |
| `node.execution_key(request, data_version)` | SHA-256：QSID、请求、数据版本 |
| 普通因子结果 RunningKey | MD5 的前 16 位：QSID、有效截面、完整时点标尺、DataVersion 或 RunID |
| 完整截面共享结果 | 执行键包含 InstanceID、截面、日期、RunID、DataVersion，再加 `section-` 前缀和分片编号 |

因子运行键包含有序截面 IDs 和完整 DTRuler；具体请求日期范围主要通过覆盖记录管理，因此同一标尺内的分段请求可以补算和复用。`context.DataVersion or context.RunID` 作为版本分量，空字符串也会退回 RunID。普通因子运行键不含 InstanceID，语义一致的独立实例在请求与版本一致时可共享结果。

原始数据缓存另按数据表的 PrepareID、字段名等业务约定组织，不应将普通因子结果键的规则套用于所有原始数据文件。

`DataVersion` 不保证数据快照真实存在，也不会自动检测数据库更新。只有调用方确保同一标签代表同一份输入数据、缓存被保留、定义和维度一致，普通因子才具备跨运行复用的条件。完整截面结果显式包含 RunID，即使 DataVersion 相同也只在本轮运行复用。

下面直接对比因子运行键的变化，不需要外部数据源。

In [6]:
from QuantStudio.Factor.Factor import FactorContext, makeFactorRunningKey

dates = [dt.datetime(2025, 1, day) for day in range(1, 4)]
context = FactorContext(DTRuler=dates, SectionIDs=["A", "B"])
first = makeFactorRunningKey(a.QSID, context=context)
context.begin_run()
assert first != makeFactorRunningKey(a.QSID, context=context)

context.DataVersion = "固定快照-v1"
first = makeFactorRunningKey(a.QSID, context=context)
context.begin_run()
assert first == makeFactorRunningKey(a.QSID, context=context)
context.DataVersion = "固定快照-v2"
assert first != makeFactorRunningKey(a.QSID, context=context)
print("未指定版本按运行隔离，固定版本允许跨运行使用同一个结果键")

未指定版本按运行隔离，固定版本允许跨运行使用同一个结果键


<a id="graph-visualization"></a>

## 计算图可视化的身份约定

`node2dict()` 生成键为 `"QSID:Name"` 的嵌套字典，重复键不再次展开依赖，叶子值为 `None`。`dict2mermaid()` 按 QSID 合并图形节点，名称作为显示文本。

因此，同 QSID 的不同 InstanceID 会画成一个节点；重复依赖边和槽位信息也不能完整保留。它适合展示计算语义，不适合直接作为实例运行状态图或可编辑连线模型。需要实例图时，应基于 InstanceID 与 `iter_dependency_edges()` 另外组织图数据。

使用示例返回[计算图框架](计算图框架.ipynb)，缓存恢复示例返回[缓存](缓存.ipynb)。